In [1]:
import os
import sys
try:
    # 1. Durum: Normal .py dosyası olarak terminalden çalıştırılıyorsa
    ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # 2. Durum: Jupyter Notebook içinden çalıştırılıyorsa (__file__ bulunamaz)
    # Jüpiter varsayılan olarak bulunduğu 'src' klasörünü dizin sayar.
    if os.path.basename(os.getcwd()) == 'src':
        ROOT_DIR = os.path.dirname(os.getcwd())
    else:
        ROOT_DIR = os.getcwd()
os.chdir(ROOT_DIR)

In [2]:
URL="https://www.teknoparkistanbul.com.tr/firmalar"

In [3]:
from bs4 import BeautifulSoup, Comment
from playwright.async_api import async_playwright

print("Playwright ve BeautifulSoup hazır!")

Playwright ve BeautifulSoup hazır!


In [4]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()

In [5]:
await page.goto(URL)
print(f"Sayfa açıldı: {URL}")

Sayfa açıldı: https://www.teknoparkistanbul.com.tr/firmalar


In [6]:
import asyncio

previous_height = await page.evaluate("document.body.scrollHeight")
print("Aşağı kaydırma işlemi başladı")

while True:
    await page.evaluate("window.scrollTo(0,document.body.scrollHeight);")
    
    try:
        await page.wait_for_function(
            f"document.body.scrollHeight > {previous_height}",timeout=4000
        )
        previous_height = await page.evaluate("document.body.scrollHeight")
    except:
        print("Sayfa sonuna ulaşıldı")
        break
await asyncio.sleep(1)

Aşağı kaydırma işlemi başladı
Sayfa sonuna ulaşıldı


In [7]:
html_content = await page.content()
soup = BeautifulSoup(html_content, "html.parser")
print ("bs4 aktarıldı sayfa")

bs4 aktarıldı sayfa


In [8]:
await browser.close()
await playwright.stop()
print("Tarayıcı Kapatıldı")

Tarayıcı Kapatıldı


In [9]:
with open("outputs/content.html","w") as f:
    f.write(html_content)

In [ ]:
import json

In [11]:
import gc
gc.collect()

205

In [12]:
companies = []
seen_keys = set()

cards = soup.find_all('div',class_='item')

for card in cards:
    name = card.find('span')
    name_text = name.get_text(strip=True) if name else "Bilinmiyor"

    desc = card.find('i')
    desc_text = desc.get_text(strip=True) if desc else "Açıklama yok"
    
    parent_a = card.find_parent('a')
    link_text = "Link yok"
    
    if parent_a:
        href = parent_a.get('href', '')
        # Eğer gerçek bir href varsa
        if href and href != "javascript:void(0)":
            link_text = f"https://www.teknoparkistanbul.com.tr{href}" if href.startswith('/') else href
        else:
            # javascript:void(0) ise yorum satırındaki linki arayalım
            comments = parent_a.find_all(string=lambda text: isinstance(text, Comment))
            found_comment_link = False
            for c in comments:
                if "http" in c:
                    link_text = c.strip()
                    found_comment_link = True
                    break
            # Yorum satırında da link yoksa alternatif olarak data attribute veya başka yerlere bakılabilir
            if not found_comment_link:
                link_text = "Link yok"

    if name_text != "Bilinmiyor" or desc_text != "Açıklama yok":
        unique_key = name_text.lower().strip()
        if unique_key not in seen_keys:
            seen_keys.add(unique_key)
            companies.append({"Firma":name_text,"Tanıtım":desc_text,"Link":link_text})

    companies = sorted(companies, key=lambda x: x['Firma'].lower())


In [13]:
companies

[{'Firma': 'A-CAR-TEC',
  'Tanıtım': 'Endüstriyel Tasarım Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'a2 Teknoloji',
  'Tanıtım': 'Endüstriyel Tasarım Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Abramak Bilgi Teknolojileri',
  'Tanıtım': 'Savunma Sanayii Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Accenture',
  'Tanıtım': 'Endüstriyel Tasarım Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Acronis', 'Tanıtım': 'Diğer', 'Link': 'Link yok'},
 {'Firma': 'Active Bioworks',
  'Tanıtım': 'Sağlık Bilimleri Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Adente',
  'Tanıtım': 'Savunma Sanayii Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Advens',
  'Tanıtım': 'Savunma Sanayii Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Aerofen',
  'Tanıtım': 'İleri Malzeme Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'Agena',
  'Tanıtım': 'Savunma Sanayii Teknolojileri',
  'Link': 'Link yok'},
 {'Firma': 'AIM Proje Yönetimi Danışmanlık Mühendislik',
  'Tanıtım': 'İle

In [14]:
with open("outputs/companies.json","w") as f:
    json.dump(companies,f,indent=4,ensure_ascii=False)

In [15]:
len(companies)

345

In [16]:
gc.collect()

33

In [17]:
without_link = [c for c in companies if c["Link"] == "Link yok"]
print(f"Toplam Firma Sayısı: {len(companies)}")
print(f"Linki Olan Firma Sayısı: {len(companies)-len(without_link)}")
print(f"Linki Olmayan Firma Sayısı: {len(without_link)}")

Toplam Firma Sayısı: 345
Linki Olan Firma Sayısı: 0
Linki Olmayan Firma Sayısı: 345
